In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("dataset.csv")

In [3]:
df['Disease_Prediction'].value_counts()

Disease_Prediction
Bovine Tuberculosis               58
Bovine Respiratory Disease        53
Equine Influenza                  45
Swine Influenza                   45
Caprine Arthritis Encephalitis    42
                                  ..
Chronic Bronchitis                 1
Tuberculosis                       1
Cryptosporidiosis                  1
Goat Pox                           1
Respiratory Syncytial Virus        1
Name: count, Length: 136, dtype: int64

In [4]:
print(df["Disease_Prediction"].value_counts()[df["Disease_Prediction"].value_counts() < 2])

Disease_Prediction
Canine Influenza               1
Canine Heartworm Disease       1
Heartworm Disease              1
Inflammatory Bowel Disease     1
Chronic Bronchitis             1
Tuberculosis                   1
Cryptosporidiosis              1
Goat Pox                       1
Respiratory Syncytial Virus    1
Name: count, dtype: int64


In [5]:
counts = df["Disease_Prediction"].value_counts()

df["Disease_Prediction"] = df["Disease_Prediction"].apply(
    lambda x: x if counts[x] > 2 else "Other"
)

In [6]:
from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(df, df["Disease_Prediction"]):
    strat_train_set = df.loc[train_index]
    strat_test_set = df.loc[test_index]

## Convert values like "102 F", "3 days" → 102 , 3

In [7]:
def extract_number(value):
    if pd.isna(value):
        return np.nan
    value = str(value)
    number = re.findall(r"\d+\.?\d*", value)

    if number:
        return float(number[0])
    else:
        return np.nan

## Convert different yes/no forms to standard values

In [8]:
def clean_yes_no(value):
    value = str(value).strip().lower()
    if value in ["yes", "y", "true", "1"]:
        return "Yes"
    if value in ["no", "n", "false", "0"]:
        return "No"
    return ""

## Standardize text (cow → Cow, german shepherd → German Shepherd)

In [9]:
def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip().title()

In [10]:
# Main preprocessing function

In [11]:
def preprocess_df(df):
    df = df.copy()

    # Convert duration to number
    if "Duration" in df.columns:
        df["Duration_Days"] = df["Duration"].apply(extract_number)
        df.drop("Duration", axis=1, inplace=True)

    # Convert numeric text columns
    numeric_cols = ["Body_Temperature", "Heart_Rate"]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].apply(extract_number)

    # Convert Yes/No columns
    yes_no_cols = [
        "Appetite_Loss",
        "Vomiting",
        "Diarrhea",
        "Coughing",
        "Labored_Breathing",
        "Lameness",
        "Skin_Lesions",
        "Nasal_Discharge",
        "Eye_Discharge",
    ]

    for col in yes_no_cols:
        if col in df.columns:
            df[col] = df[col].apply(clean_yes_no)

    # Clean text columns
    text_cols = [
        "Animal_Type",
        "Breed",
        "Gender",
        "Symptom_1",
        "Symptom_2",
        "Symptom_3",
        "Symptom_4",
    ]

    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].apply(clean_text)

    return df

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report
import joblib
from preprocess_utils import preprocess_df

In [13]:
MODEL_PATH = "model.pkl"

In [14]:
df = pd.read_csv("dataset.csv")

In [15]:
import re

In [16]:
df = preprocess_df(df)

In [17]:
target = "Disease_Prediction"
if target not in df.columns:
    raise ValueError("Disease_Prediction column not found in dataset")

In [18]:
X = df.drop(columns=[target])         #Features
y = df[target]                        #Labels

In [19]:
numeric_features = [
        "Age",
        "Weight",
        "Duration_Days",
        "Body_Temperature",
        "Heart_Rate",
    ]

In [20]:
categorical_features = [c for c in X.columns if c not in numeric_features]

In [21]:
numeric_transformer = Pipeline(
    steps = [
            ("imputer", SimpleImputer(strategy="median"))
    ]
    )

In [22]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

In [23]:
preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

# Using Decision Tree algo:

In [24]:
from sklearn.tree import DecisionTreeClassifier

In [25]:
model = DecisionTreeClassifier()

In [26]:
clf = Pipeline(
        steps=[
            ("prep", FunctionTransformer(preprocess_df, validate=False)),
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

In [28]:
class_counts = y.value_counts()
if class_counts.min() < 2:
        # Too few samples in some classes for stratified split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

In [29]:
clf.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 FunctionTransformer(func=<function preprocess_df at 0x00000260E7A42CA0>)),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age', 'Weight',
                                                   'Duration_Days',
                                                   'Body_Temperature',
                                                   'Heart_Rate']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Animal_Type', 'Breed',
                                                   'Gender', 'Symptom_1',
                                                   'Symptom_2', 'Symptom_3',
                                                   'Symptom_4', 'Appetite_Loss',
                                                   'Vomiting', 'Diarrhea',
                                                   'Coughing',
                                                   'Labored_Breathing',
                                                   'Lameness', 'Skin_Lesions',
                                                   'Nasal_Discharge',
                                                   'Eye_Discharge'])])),
                ('model', DecisionTreeClassifier())])

In [30]:
y_pred = clf.predict(X_test)

In [34]:
acc = accuracy_score(y_test, y_pred)

In [35]:
acc

0.8633333333333333

In [36]:
print(f"Accuracy: {acc:.4f}")

Accuracy: 0.8633


In [37]:
print(f"Classification Report :",classification_report(y_test, y_pred, zero_division=0))

Classification Report :                                                precision    recall  f1-score   support

               Actinobacillus Pleuropneumonia       0.50      1.00      0.67         4
                          Actinobacillus Suis       1.00      1.00      1.00         1
                          African Swine Fever       1.00      1.00      1.00         1
                            Allergic Rhinitis       1.00      1.00      1.00         1
                          Blue Tongue Disease       0.50      1.00      0.67         1
                            Blue Tongue Virus       1.00      1.00      1.00         2
                                   Bluetongue       0.33      0.67      0.44         3
                             Bluetongue Virus       1.00      0.50      0.67         2
                             Bovine Influenza       1.00      1.00      1.00         2
                       Bovine Johne's Disease       1.00      1.00      1.00         4
                  

# Using Random Forest:

In [38]:
from sklearn.ensemble import RandomForestClassifier

In [39]:
model = RandomForestClassifier()

In [40]:
clf = Pipeline(
        steps=[
            ("prep", FunctionTransformer(preprocess_df, validate=False)),
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

In [41]:
class_counts = y.value_counts()
if class_counts.min() < 2:
        # Too few samples in some classes for stratified split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

In [42]:
clf.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 FunctionTransformer(func=<function preprocess_df at 0x00000260E7A42CA0>)),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age', 'Weight',
                                                   'Duration_Days',
                                                   'Body_Temperature',
                                                   'Heart_Rate']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Animal_Type', 'Breed',
                                                   'Gender', 'Symptom_1',
                                                   'Symptom_2', 'Symptom_3',
                                                   'Symptom_4', 'Appetite_Loss',
                                                   'Vomiting', 'Diarrhea',
                                                   'Coughing',
                                                   'Labored_Breathing',
                                                   'Lameness', 'Skin_Lesions',
                                                   'Nasal_Discharge',
                                                   'Eye_Discharge'])])),
                ('model', RandomForestClassifier())])

In [43]:
y_pred = clf.predict(X_test)

In [44]:
acc = accuracy_score(y_test, y_pred)

In [45]:
acc

0.9

In [46]:
print(f"Accuracy: {acc:.4f}")

Accuracy: 0.9000


In [47]:
print(f"Accuracy: {acc:.4f}")

Accuracy: 0.9000


In [48]:
# Classification Report:

In [49]:
print(classification_report(y_test, y_pred, zero_division=0))

                                               precision    recall  f1-score   support

               Actinobacillus Pleuropneumonia       0.67      1.00      0.80         4
                          Actinobacillus Suis       1.00      1.00      1.00         1
                          African Swine Fever       1.00      1.00      1.00         1
                            Allergic Rhinitis       1.00      1.00      1.00         1
                          Blue Tongue Disease       0.20      1.00      0.33         1
                            Blue Tongue Virus       1.00      1.00      1.00         2
                                   Bluetongue       1.00      1.00      1.00         3
                             Bluetongue Virus       1.00      0.50      0.67         2
                             Bovine Influenza       1.00      1.00      1.00         2
                       Bovine Johne's Disease       1.00      1.00      1.00         4
                        Bovine Leukemia Vi

In [50]:
## Difference in the o/p

In [51]:
import pandas as pd

In [52]:
df = pd.read_csv("output.csv")

In [53]:
df.head()

,Expected,Predicted
0,Caprine Arthritis Encephalitis,Caprine Arthritis Encephalitis
1,Porcine Epidemic Diarrhea Virus,Porcine Epidemic Diarrhea Virus
2,Bovine Respiratory Disease,Bovine Respiratory Disease
3,Canine Leptospirosis,Canine Leptospirosis
4,Canine Hepatitis,Canine Hepatitis


In [54]:
# error = df["Expected"] != df["Predicted"]

In [55]:
df[df["Expected"] != df["Predicted"]]

,Expected,Predicted
39,Swine Fever,Swine Influenza
64,Tuberculosis,Bovine Respiratory Disease
84,Swine Dysentery,Swine Influenza
101,Feline Leukemia,Conjunctivitis
106,Feline Herpesvirus,Feline Calicivirus
107,Bovine Tuberculosis,Bovine Respiratory Disease
118,Bovine Tuberculosis,Bovine Respiratory Disease
135,Equine Herpesvirus,Equine Leptospirosis
192,Kennel Cough,Canine Cough
195,Equine Influenza,West Nile Virus
